In [1]:
%load_ext autoreload
%autoreload 2

In [2]:


from src.main.python.iSel import cnn, enn, icf, lssm, lsbo, drop3, ldis, cdis, xldis, psdsp, ib3, cis, egdis, e2sc, biois
from src.main.python.utils.general import get_data
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [3]:

def get_selector(method: str):
    print(f"IS-Method: {method}")
    #Baselines
    if method == 'cnn':     return cnn.CNN()
    if method == 'enn':     return enn.ENN()
    if method == 'icf':     return icf.ICF()
    if method == 'lssm':    return lssm.LSSm()
    if method == 'lsbo':    return lsbo.LSBo()
    if method == 'drop3':   return drop3.DROP3()
    if method == 'ldis':    return ldis.LDIS()
    if method == 'cdis':    return cdis.CDIS()
    if method == 'xldis':   return xldis.XLDIS()
    if method == 'psdsp':   return psdsp.PSDSP()
    if method == 'ib3':     return ib3.IB3()
    if method == 'egdis':   return egdis.EGDIS()
    if method == 'cis':     return cis.CIS(task="atc")
    #proposed framework
    if method == 'e2sc-1':   return e2sc.E2SC(alphaMode="exact", betaMode='iterative')
    if method == 'e2sc-2':   return e2sc.E2SC(alphaMode="approximated", betaMode='heuristic')
    if method == 'bio-is':   return biois.BIOIS(beta=0.25, theta=0.50)
    return None


# Opening data - aisopos_ntua_2L dataset

In [4]:
inputdir = "resources/datasets/aisopos_ntua_2L/tfidf/"

X_train, y_train, X_test, y_test, _ = get_data(inputdir, f=0)

# Example CNN - Selecting Instances

In [5]:
#selector = e2sc.E2SC(alphaMode="approximated", beta=0.15)
#selector = e2sc.E2SC(alphaMode="exact", betaMode='iterative')
#selector = get_selector(method="e2sc-1")
#selector = get_selector(method="e2sc-2")
selector = get_selector(method="bio-is")
selector.fit(X_train, y_train)
idx = selector.sample_indices_
#print(idx)
X_train_selected, y_train_selected =  X_train[idx], y_train[idx]
selector.reduction_

IS-Method: bio-is
fitting_alpha_by_lr_default
LogisticRegression(n_jobs=-1)
LogisticRegression(n_jobs=-1)
LogisticRegression(n_jobs=-1)
LogisticRegression(n_jobs=-1)
LogisticRegression(n_jobs=-1)
Micro: 0.596
Macro: 0.44308432034231016
identifyNoiseByLowerNNEntropy


0.44799999999999995

In [6]:
import pandas as pd

# Displaying the head of the results as a DataFrame (similar to the persisted format)
df_results = pd.DataFrame([{
    'train_idxs': selector.sample_indices_,
    'entropy': getattr(selector, 'entropy_', None)
}])

print("Selection Results Summary:")
df_results.head()

Selection Results Summary:


,train_idxs,entropy
0,"[0, 3, 4, 5, 7, 11, 12, 13, 15, 16, 19, 20, 23...","[0.6892849495177986, 0.680629523690004, 0.6802..."


# Example CNN - Comparing Classifiers

In [7]:
clf = KNeighborsClassifier(n_neighbors=1)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"NoSel: {acc}")

clf = KNeighborsClassifier(n_neighbors=1)
clf.fit(X_train_selected, y_train_selected)

y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"CNN: {acc}")


NoSel: 0.75
CNN: 0.6785714285714286


# make_classification Example

In [8]:
from collections import Counter
from sklearn.datasets import make_classification
from src.main.python.iSel import e2sc
X, y = make_classification(n_classes=2, class_sep=2, weights=[0.1, 0.9], n_informative=3, n_redundant=1, flip_y=0, n_features=20, n_clusters_per_class=1, n_samples=1000, random_state=10)
print('Original dataset shape %s' % Counter(y))



Original dataset shape Counter({1: 900, 0: 100})


In [9]:
selector = e2sc.E2SC()
selector.fit(X, y)
idx = selector.sample_indices_
X_train_selected, y_train_selected =  X[idx], y[idx]
print('Resampled dataset shape %s' % Counter(y_train_selected))


Resampled dataset shape Counter({1: 37, 0: 13})


In [10]:
from run_generateSplit import main as run_split_main
from pathlib import Path
from datetime import datetime

# Setup paths
datain = Path("resources/datasets").resolve()
timestamp = datetime.now().strftime("%Y-%m-%d_%H_%M")
out = Path("output") / timestamp
out.mkdir(parents=True, exist_ok=True)

# Define dataset and method
dataset = "aisopos_ntua_2L"
method = "bio-is"

args_list = [
    "-d", dataset,
    "-m", method,
    "--datain", str(datain),
    "--out", str(out)
]

# Call run_generateSplit directly with debug=True
print(f"Running {method} on {dataset}...")
run_split_main(args_list, debug=True)

Running bio-is on aisopos_ntua_2L...
Namespace(datain='C:\\Users\\Diogo Neiss\\Documents\\01_LLM_sustentavel\\biobj-instance-selection-and-cl\\resources\\datasets', dataset='aisopos_ntua_2L', filename='output\\2026-05-14_23_32\\selection\\aisopos_ntua_2L\\saida_bio-is', folds=10, inputdir='C:\\Users\\Diogo Neiss\\Documents\\01_LLM_sustentavel\\biobj-instance-selection-and-cl\\resources\\datasets\\aisopos_ntua_2L\\tfidf', inputrep='tfidf', method='bio-is', out='output\\2026-05-14_23_32', outputdir='output\\2026-05-14_23_32\\selection\\aisopos_ntua_2L', overwrite=0, save=True, splitdir='C:\\Users\\Diogo Neiss\\Documents\\01_LLM_sustentavel\\biobj-instance-selection-and-cl\\resources\\datasets\\aisopos_ntua_2L\\splits', start_time='14-05-2026 23:32:59')
Criando saida output\2026-05-14_23_32\selection\aisopos_ntua_2L
C:\Users\Diogo Neiss\Documents\01_LLM_sustentavel\biobj-instance-selection-and-cl\resources\datasets\aisopos_ntua_2L\splits\split_10.pkl
Fold 0
fitting_alpha_by_lr_default
Log